# Safety Properties for RL

## Safety Properties for Reinforcement Learning

RL agents trained via reward maximization can learn unsafe policies. Safety in RL requires guarantees that the agent never enters dangerous states.

**Shield synthesis**: A safety shield is a runtime monitor that overrides the agent's unsafe actions with safe alternatives. The shield is computed from a formal safety specification.

**Safe exploration**: During training, the agent must avoid dangerous states while exploring the environment. Constrained MDP formulations add safety constraints to the reward objective.

**Lyapunov-based safety**: Neural Lyapunov functions certify that the system state converges to a safe set from any initial condition.

In [ ]:
import numpy as np
from typing import List, Dict, Tuple, Optional

np.random.seed(42)

# Simple gridworld with safety constraints
# Grid: 5x5, agent starts at (0,0), goal at (4,4)
# Unsafe states: row 2 (hazards)

GRID_SIZE = 5
UNSAFE_ROW = 2
ACTIONS = ['up', 'down', 'left', 'right']

def step(state, action):
    r, c = state
    if action == 'up': r = max(0, r-1)
    elif action == 'down': r = min(GRID_SIZE-1, r+1)
    elif action == 'left': c = max(0, c-1)
    elif action == 'right': c = min(GRID_SIZE-1, c+1)
    return (r, c)

def is_unsafe(state): return state[0] == UNSAFE_ROW
def is_goal(state): return state == (4, 4)

class SafetyShield:
    def __init__(self):
        # Precompute safe actions for each state
        self.safe_actions = {}
        for r in range(GRID_SIZE):
            for c in range(GRID_SIZE):
                state = (r, c)
                safe = [a for a in ACTIONS if not is_unsafe(step(state, a))]
                self.safe_actions[state] = safe if safe else ACTIONS
    
    def shield(self, state, proposed_action):
        next_state = step(state, proposed_action)
        if is_unsafe(next_state):
            safe = self.safe_actions[state]
            return np.random.choice(safe), True  # (action, was_overridden)
        return proposed_action, False

# Random agent without shield
def run_episode(use_shield=False, max_steps=50):
    shield = SafetyShield() if use_shield else None
    state = (0, 0)
    safety_violations = 0
    interventions = 0
    
    for _ in range(max_steps):
        action = np.random.choice(ACTIONS)
        if use_shield:
            action, intervened = shield.shield(state, action)
            interventions += int(intervened)
        state = step(state, action)
        if is_unsafe(state):
            safety_violations += 1
        if is_goal(state):
            break
    
    return state == (4, 4), safety_violations, interventions

# Compare with and without shield
N_EPISODES = 200
results_no_shield = [run_episode(False) for _ in range(N_EPISODES)]
results_with_shield = [run_episode(True) for _ in range(N_EPISODES)]

print('Safety Shield Results:')
print(f'{'Metric':<25} {'No Shield':>10} {'With Shield':>12}')
print('-' * 50)
print(f'{'Goal reached (%)'::<25} '
      f'{sum(r[0] for r in results_no_shield)/N_EPISODES:>10.3f} '
      f'{sum(r[0] for r in results_with_shield)/N_EPISODES:>12.3f}')
print(f'{'Safety violations'::<25} '
      f'{sum(r[1] for r in results_no_shield)/N_EPISODES:>10.2f} '
      f'{sum(r[1] for r in results_with_shield)/N_EPISODES:>12.2f}')
print(f'{'Shield interventions'::<25} '
      f'{'N/A':>10} '
      f'{sum(r[2] for r in results_with_shield)/N_EPISODES:>12.2f}')
